# From raw files to ODS tables

The capture jobs store every file exactly as the source served it. Nothing reads those files yet. This notebook designs the first warehouse layer, **ODS**: the raw files as BigQuery tables. It is the spec for the Dagster ODS assets of phase 1, steps 3 to 5.

| Question | Why it matters |
| --- | --- |
| What does an ODS row look like, and what does it carry besides the data? | One contract for every table, so lineage and the time machine work the same everywhere |
| Where does ODS stop and DWD begin? | Rules about meaning (units, slot times, `known_at`) live in one place |
| What shape should each source take? | CSV and JSON sources look nothing alike |
| How do repeated captures of one file become versions? | The time machine needs "what was known at `T`" |
| Do the parsers survive every file we hold, not just one sample? | Old and new files differ; a parser proven on one file proves little |
| How does this map onto Dagster? | Partitions, re-parsing, checks |

It reads the **prod raw bucket, read-only**, through `jtf.storage`, and caches every file under `data/raw-cache/`. Raw files never change, so the cache never goes stale. Claims are labelled as in the earlier notebooks: **Observed** (a cell shows it) or **Source** (quoted from the publisher, with a link).

## 0. Setup

In [1]:
import io
import json
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from email.utils import parsedate_to_datetime
from pathlib import Path

import numpy as np
import pandas as pd

from jtf.ingest.sources import SOURCES, WEATHER_POINTS
from jtf.storage import open_store

RAW_URL = "gs://jepx-tokyo-forecast-prod-raw"
store = open_store(RAW_URL)
CACHE = Path("../data/raw-cache")
pd.set_option("display.width", 160, "display.max_columns", 30)


def read(path: str) -> bytes:
    """Read a raw file through the local cache."""
    local = CACHE / path
    if not local.exists():
        local.parent.mkdir(parents=True, exist_ok=True)
        local.write_bytes(store.get(path))
    return local.read_bytes()


def sidecar(path: str) -> dict:
    return json.loads(read(path + ".meta.json"))


def decode(raw: bytes) -> str:
    """TEPCO and JEPX files are UTF-8 or CP932 (Shift_JIS), varying by file (notebook 01)."""
    try:
        return raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        return raw.decode("cp932")


def headers_of(meta: dict) -> dict:
    return {k.lower(): v for k, v in meta.get("response_headers", {}).items()}


paths = [p for p in store.list("") if not p.endswith(".meta.json")]
with ThreadPoolExecutor(16) as pool:
    metas = list(pool.map(sidecar, paths))
files = pd.DataFrame(
    {
        "path": paths,
        "source": [p.split("/")[0] for p in paths],
        "extraction_date": [p.split("/")[1] for p in paths],
        "filename": [m["filename"] for m in metas],
        "load_type": [m.get("load_type", "daily") for m in metas],
        "fetched_at": pd.to_datetime([m["fetched_at"] for m in metas]),
        "http_status": [m.get("http_status") for m in metas],
        "gap": [bool(m.get("gap")) for m in metas],
        "bytes": [m["size_bytes"] for m in metas],
        "sha256": [m["sha256"] for m in metas],
        "last_modified": [headers_of(m).get("last-modified") for m in metas],
    }
)
files["registered"] = files["source"].isin(SOURCES)
files["fetched_jst"] = files["fetched_at"].dt.tz_convert("Asia/Tokyo").dt.strftime("%m-%d %H:%M")
print(f"{len(files)} raw files read from {RAW_URL}")

947 raw files read from gs://jepx-tokyo-forecast-prod-raw


## 1. What is in the raw bucket

**Step: take stock** before designing anything: which prefixes, how many files of each load type, how big, fetched when. The sidecars (`.meta.json`) hold all of it, so no data file has to be opened yet.

In [2]:
print("extraction dates:", sorted(files["extraction_date"].unique()))
print(
    "prefixes that are not registered sources:",
    sorted(files.loc[~files["registered"], "source"].unique()),
)
files.groupby(["source", "load_type"]).agg(
    files=("path", "count"),
    distinct=("sha256", "nunique"),
    mb=("bytes", lambda b: round(b.sum() / 1e6, 1)),
    first_jst=("fetched_jst", "min"),
    last_jst=("fetched_jst", "max"),
)

extraction dates: ['2026-11-15']
prefixes that are not registered sources: ['tepco_forecast_next_day', 'tepco_sameday']


files  distinct    mb    first_jst     last_jst
source                       load_type                                                 
jepx_spot                    daily          3         1   4.4  11-15 11:08  11-15 12:24
                             initial       10        10  22.0  11-15 13:52  11-15 13:53
openmeteo_live               daily          2         2   0.2  11-15 11:36  11-15 12:24
openmeteo_previous_runs      initial       35        35  59.9  11-15 13:52  11-15 14:01
openmeteo_single_runs        initial      819       819  61.9  11-15 14:01  11-15 14:41
tepco_actuals                daily          6         4   0.4  11-15 11:04  11-15 12:24
                             initial       41        41   8.4  11-15 13:52  11-15 13:54
                             recapture      1         1   0.1  11-15 14:02  11-15 14:02
tepco_area_actuals_today     daily          1         1   0.0  11-15 12:24  11-15 12:24
tepco_area_bg_plan           daily          1         1   0.0  11-15 12:24  11-15 12:24
tepco_area_bg_plan_next_day  daily          1         1   0.0  11-15 12:24  11-15 12:24
tepco_area_forecast          daily          1         1   0.0  11-15 12:24  11-15 12:24
tepco_area_forecast_next_day daily          1         1   0.0  11-15 12:24  11-15 12:24
tepco_demand_outlook         daily          1         1   0.0  11-15 12:24  11-15 12:24
tepco_forecast_next_day      daily          4         1   0.0  11-15 11:08  11-15 11:35
tepco_sameday                daily         20        10   0.1  11-15 11:04  11-15 11:36

**What it shows.** **Observed:**

- 947 files under 13 prefixes: the 11 registered sources and 2 legacy ones. The history (`initial`) is 905 of them, mostly Open-Meteo single runs (819 so far; tomorrow's resume adds about 550)
- **Every file has the extraction date 2026-11-15**, including ten years of JEPX prices. The extraction date says when we fetched a file, never what period it covers. Anything partitioned by extraction date (the raw layout, Dagster's ODS partitions) puts the whole history in one partition, and that is fine: it is one load
- `distinct` < `files` for the repeated daily captures: the same content fetched again. Those copies are kept; section 4 shows why they cost nothing downstream
- `tepco_forecast_next_day` and `tepco_sameday` are the test captures from this morning, before capture was split per source (see `jtf/src/jtf/ingest/README.md`). They are not registered sources

**Learned → design:** ODS is built from the **source registry**, not from whatever prefixes exist. The 24 legacy files stay in the bucket (nothing can delete them, by design) and are never parsed.

## 2. The proposed ODS contract

ODS (operational data store) is the first warehouse layer. The rule that keeps it simple: **ODS changes the format, never the meaning.**

| ODS does | ODS leaves to DWD (dbt) |
| --- | --- |
| Decode the file (encoding, CSV or JSON), one ODS row per data row | Convert units (kWh per slot → MW) |
| Rename Japanese headers to fixed ASCII names, one name list per table | Turn slot labels (`0:00`, `0000`, code 1–48) into timestamps |
| Type the values (numbers, dates) | Decide which capture's value is the version in force |
| Undo name-packing: Open-Meteo hides the model and lead inside column names; ODS makes them columns | Work out `known_at` |
| Add lineage columns | Fill holes, choose fallbacks |
| Check the file's shape and **refuse it** if the shape changed | Plausibility checks about the world (solar at night, under capacity) |

Why split it there: every rule about meaning lives in dbt SQL, tested and versioned in one place, and changing one re-runs dbt, not the parsers. ODS can be rebuilt from the raw bucket at any time and gives the same rows.

**Lineage columns**, the same on every ODS table:

| Column | Comes from | Why |
| --- | --- | --- |
| `source_file` | The raw path | The exact file a row came from; joins to the file manifest (section 8) |
| `extraction_date` | Sidecar | The partition, the same as the raw layout and the Dagster partition |
| `fetched_at` | Sidecar | When we had the file: the basis of `known_at` for captures |
| `load_type` | Sidecar | `daily`, `initial` or `recapture` |
| `source_time` | The update time written in the file (TEPCO area files), else the `Last-Modified` header, else null | When the publisher says the content was written |
| `row_number` | The position in the file (CSV tables) | Ties a row to its line; makes duplicate checks exact |
| `parser_version` | The parser | Which parser wrote the row; a re-parse adds rows under a new version |
| `ingested_at`, `batch_id` | The load: BigQuery load time, Dagster run ID | When, and by which run, the row was loaded |

This differs from the draft contract in the phase-1 README in two places, each tested below. ODS has **no `known_at`**: section 4 shows it depends on other captures of the file, which a parser of one file cannot see. ODS also has **no shared `valid_time` column**: each table keeps the source's own time columns, and DWD turns them into slot starts.

## 3. TEPCO actuals: the target

**Source:** the monthly files listed on [TEPCO's area supply-and-demand page](https://www.tepco.co.jp/forecast/html/area_jukyu-j.html). Notebook 01 found the first row is a unit row, `単位[MW平均]` ("unit: MW, averaged"), then one header row, then one row per 30-minute slot.

**Step: write the parser the way production would**, failing loudly on anything unexpected: a different unit row, or a column it has no ASCII name for. A column that is missing becomes nulls, so an older file with fewer columns still loads. Then parse the newest November capture.

In [3]:
PARSER_VERSION = "draft-1"

TEPCO_ACTUALS_COLUMNS = {
    "DATE": "date",
    "TIME": "time",
    "エリア需要": "area_demand",
    "原子力": "nuclear",
    "火力(LNG)": "thermal_lng",
    "火力(石炭)": "thermal_coal",
    "火力(石油)": "thermal_oil",
    "火力(その他)": "thermal_other",
    "水力": "hydro",
    "地熱": "geothermal",
    "バイオマス": "biomass",
    "太陽光発電実績": "solar",
    "太陽光出力制御量": "solar_curtailed",
    "風力発電実績": "wind",
    "風力出力制御量": "wind_curtailed",
    "揚水": "pumped_storage",
    "蓄電池": "battery",
    "連系線": "interconnectors",
    "その他": "other",
    "合計": "total",
}
TEPCO_ACTUALS_VALUES = [c for c in TEPCO_ACTUALS_COLUMNS.values() if c not in ("date", "time")]


def lineage(path: str, meta: dict, source_time: pd.Timestamp | None = None) -> dict:
    """The columns every ODS row carries besides its data."""
    last_modified = headers_of(meta).get("last-modified")
    if source_time is None and last_modified:
        source_time = pd.Timestamp(parsedate_to_datetime(last_modified))
    return {
        "source_file": path,
        "extraction_date": meta["extraction_date"],
        "fetched_at": pd.Timestamp(meta["fetched_at"]),
        "load_type": meta.get("load_type", "daily"),
        "source_time": source_time if source_time is not None else pd.NaT,
        "parser_version": PARSER_VERSION,
    }


def parse_tepco_actuals(path: str) -> pd.DataFrame:
    lines = decode(read(path)).splitlines()
    unit = lines[0].split(",")[0]
    if unit != "単位[MW平均]":
        raise ValueError(f"unit row is {unit!r}")
    df = pd.read_csv(io.StringIO("\n".join(lines[1:])), dtype=str)
    unknown = set(df.columns) - set(TEPCO_ACTUALS_COLUMNS)
    if unknown:
        raise ValueError(f"no ASCII name for columns {sorted(unknown)}")
    df = df.rename(columns=TEPCO_ACTUALS_COLUMNS).reindex(
        columns=list(TEPCO_ACTUALS_COLUMNS.values())
    )
    df.insert(0, "row_number", range(1, len(df) + 1))
    df = df.dropna(subset=["area_demand"])
    for column in TEPCO_ACTUALS_VALUES:
        df[column] = pd.to_numeric(df[column])
    return df.assign(**lineage(path, sidecar(path)))


november = files[files["filename"] == "eria_jukyu_202611_03.csv"].sort_values("fetched_at")
latest_path = november["path"].iloc[-1]
actuals = parse_tepco_actuals(latest_path)
data_lines = len(decode(read(latest_path)).splitlines()) - 2
print(latest_path, "->", len(actuals), "rows from", data_lines, "data lines")
actuals.head(2).T

tepco_actuals/2026-11-15/20261115T050248Z_eria_jukyu_202611_03.csv -> 699 rows from 699 data lines


,0,1
row_number,1,2
date,2026/11/1,2026/11/1
time,0:00,0:30
area_demand,21530,20770
nuclear,1309,1308
thermal_lng,9602,8687
thermal_coal,2612,2681
thermal_oil,233,232
thermal_other,730,739
hydro,1071,1066


**Step: per-file checks.** These become the Dagster asset checks for the table: every complete day has 48 slots, no slot appears twice, solar is never negative. One more, out of interest: does the supply total match demand?

In [4]:
day = pd.to_datetime(actuals["date"], format="%Y/%m/%d")
per_day = day.value_counts().sort_index()
print(
    "slots per complete day:",
    sorted(per_day.iloc[:-1].unique()),
    "| slots so far on the last day:",
    per_day.iloc[-1],
)
print("duplicate (date, time):", actuals.duplicated(["date", "time"]).sum())
print("solar MW:", actuals["solar"].min(), "to", actuals["solar"].max())
balance = actuals["total"] - actuals["area_demand"]
print(
    f"total == demand in {balance.eq(0).mean():.0%} of slots; largest gap {balance.abs().max()} MW"
)
last = actuals.iloc[-1]
print(
    f"last slot starts {last['date']} {last['time']} JST; "
    f"file Last-Modified {november['last_modified'].iloc[-1]}"
)

slots per complete day: [np.int64(48)] | slots so far on the last day: 27
duplicate (date, time): 0
solar MW: 0 to 13897
total == demand in 41% of slots; largest gap 3 MW
last slot starts 2026/11/15 13:00 JST; file Last-Modified Sun, 15 Nov 2026 04:40:04 GMT


**What it shows.** **Observed:**

- 699 rows, one per data line: 14 complete days of 48 slots, plus 27 slots of today. Future slots are simply absent, not blank
- No duplicate slots; solar from 0 at night to 13,897 MW at midday
- `total` (the sum of the supply columns) equals `area_demand` exactly in 41% of slots and is never more than 3 MW off: rounding of the components. A balance check needs a tolerance of a few MW, not equality
- The last slot starts at 13:00 JST and ends at 13:30; the file was written at 13:40 JST (04:40 GMT). TEPCO publishes a slot about **10 minutes after it ends**

**Learned → design:** `ods.tepco_actuals` keeps TEPCO's own `date` and `time` strings and MW values unchanged. The history rule "slot end + 1 h" (PRD) is conservative by about 50 minutes, which is the right direction for a rule that stands in for missing captures.

## 4. Versions: one file, captured seven times

The daily job fetches the whole current-month file every day, and TEPCO keeps rewriting it: new slots every half hour and, notebook 01 showed, silent corrections weeks later. Today the file was captured seven times (test runs and the re-capture). That is a small, real example of the problem the time machine solves: **what did we know at time `T`?**

**Step: parse every capture.** ODS keeps them all: seven files in, seven sets of rows out.

In [5]:
ods_november = pd.concat([parse_tepco_actuals(p) for p in november["path"]], ignore_index=True)
print(
    f"{len(november)} captures, {november['sha256'].nunique()} distinct contents "
    f"-> {len(ods_november)} ODS rows"
)
november.assign(
    sha256=november["sha256"].str[:10],
    rows=ods_november.groupby("source_file").size().reindex(november["path"]).to_numpy(),
)[["fetched_jst", "load_type", "bytes", "sha256", "rows", "last_modified"]]

7 captures, 5 distinct contents -> 4864 ODS rows


,fetched_jst,load_type,bytes,sha256,rows,last_modified
869,11-15 11:04,daily,62952,d041f00d52,693,"Sun, 15 Nov 2026 01:40:04 GMT"
870,11-15 11:06,daily,62952,d041f00d52,693,"Sun, 15 Nov 2026 01:40:04 GMT"
871,11-15 11:11,daily,63042,b254ee8e13,694,"Sun, 15 Nov 2026 02:10:03 GMT"
872,11-15 11:13,daily,63042,b254ee8e13,694,"Sun, 15 Nov 2026 02:10:03 GMT"
873,11-15 11:36,daily,63133,dcebd237ab,695,"Sun, 15 Nov 2026 02:35:00 GMT"
874,11-15 12:24,daily,63223,faa56e0474,696,"Sun, 15 Nov 2026 03:10:04 GMT"
916,11-15 14:02,recapture,63495,51716d5764,699,"Sun, 15 Nov 2026 04:40:04 GMT"


**Step: turn captures into versions.** Unpivot to one value per (slot, measure, capture), sort by capture time, and keep a value only when it differs from the previous capture's value for the same slot and measure. What remains is every version that ever existed, stamped with the capture that first showed it.

In [6]:
KEY = ["date", "time", "measure"]
long = ods_november.melt(
    id_vars=["date", "time", "fetched_at", "source_file"],
    value_vars=TEPCO_ACTUALS_VALUES,
    var_name="measure",
).sort_values([*KEY, "fetched_at"])
# A value is a new version when it differs from the previous capture's; NaN to NaN is no change
# (in SQL: IS DISTINCT FROM).
previous = long.groupby(KEY)["value"].shift()
is_new = long["value"].ne(previous) & ~(long["value"].isna() & previous.isna())
versions = long[is_new].copy()
print(f"ODS long rows {len(long):,} -> versions {len(versions):,}")
print(
    "(slot, measure) pairs whose value changed after first appearing:",
    versions.groupby(KEY).size().gt(1).sum(),
)
versions.groupby("fetched_at").size().rename("new versions").to_frame()

ODS long rows 87,552 -> versions 12,582
(slot, measure) pairs whose value changed after first appearing: 0


,new versions
fetched_at,
2026-11-15 02:04:16.643393+00:00,12474
2026-11-15 02:11:46.678216+00:00,18
2026-11-15 02:36:23.477086+00:00,18
2026-11-15 03:24:01.537490+00:00,18
2026-11-15 05:02:48.862189+00:00,54


**Step: stamp `known_at` on each version.** A capture proves a value was public at `fetched_at`. Before capture started there is no proof, so a history rule stands in (PRD: slot end + 1 hour for TEPCO actuals). The question is which rule applies to which version, and the November file shows why `load_type` alone can't answer it: the first **daily** capture already held 14 days of slots from before capture began.

The rule that works: for the **first version** of a slot, if the history rule's time is before capture start, use the rule (`basis = rule`); otherwise use the capture (`basis = captured`). A **later version** (a correction) is known only from the capture that first showed it.

In [7]:
CAPTURE_START = files[files["load_type"] == "daily"].groupby("source")["fetched_at"].min()


def stamp_known_at(
    versions: pd.DataFrame, key: list[str], rule_time: pd.Series, capture_start
) -> pd.DataFrame:
    first = ~versions.duplicated(key)
    use_rule = first & (rule_time < capture_start)
    return versions.assign(
        known_at=rule_time.where(use_rule, versions["fetched_at"]),
        basis=np.where(use_rule, "rule", "captured"),
    )


slot_start = pd.to_datetime(
    versions["date"] + " " + versions["time"], format="%Y/%m/%d %H:%M"
).dt.tz_localize("Asia/Tokyo")
stamped = stamp_known_at(
    versions, KEY, slot_start + pd.Timedelta(minutes=90), CAPTURE_START["tepco_actuals"]
)
print("capture start:", CAPTURE_START["tepco_actuals"].tz_convert("Asia/Tokyo"))
print(stamped["basis"].value_counts().to_string())
solar = stamped.assign(slot_start=slot_start)[lambda d: d["measure"] == "solar"].sort_values(
    "slot_start"
)
handover = solar[solar["slot_start"].between("2026-11-15 09:00+09:00", "2026-11-15 11:30+09:00")]
handover[["date", "time", "value", "known_at", "basis"]].assign(
    known_at=lambda d: d["known_at"].dt.tz_convert("Asia/Tokyo").dt.strftime("%m-%d %H:%M")
)

capture start: 2026-11-15 11:04:16.643393+09:00
basis
rule        12456
captured      126


,date,time,value,known_at,basis
44466,2026/11/15,9:00,1485,11-15 10:30,rule
44467,2026/11/15,9:30,2187,11-15 11:00,rule
44468,2026/11/15,10:00,2963,11-15 11:04,captured
45855,2026/11/15,10:30,3317,11-15 11:11,captured
47244,2026/11/15,11:00,2937,11-15 11:36,captured
47940,2026/11/15,11:30,2252,11-15 12:24,captured


**What it shows.** **Observed:**

- 7 captures, 5 distinct contents, 4,864 ODS rows. Two captures repeated the previous one byte for byte
- 87,552 values become 12,582 versions: 12,474 (693 slots × 18 measures) from the first capture, then exactly 18 per later capture (one new slot) and 54 from the re-capture (three new slots). **No value changed today.** Corrections arrive weeks later (notebook 01); the same code catches them when they do
- Identical captures add no versions. That is why keeping every daily copy costs nothing downstream
- 12,456 versions carry the rule's time and 126 (7 slots × 18 measures) a capture's. The table shows the hand-over. The 9:30 slot's rule time (11:00) is before capture start (11:04), so the rule stands. The 10:00 slot's rule time (11:30) is after it, so its `known_at` is the 11:04 capture that first showed it, and each later slot is stamped by the capture that first showed it

**Learned → design:**

- **Versioning and `known_at` belong in DWD**, not ODS. Both compare a row with other captures of the same file, which one file's parser never sees
- The `known_at` rule is per **version**, keyed on capture start, not on `load_type`. The DWD model keeps one row per slot and version, with `known_at` and `basis`; `as_of(T)` picks, per slot, the latest version with `known_at ≤ T`
- `basis` lets a time-machine run report its fidelity from the rows it actually used: exact when every row was `captured`

## 5. JEPX spot prices

**Source:** the yearly `spot_summary_{FY}.csv` files on the [JEPX spot market page](https://www.jepx.jp/electricpower/market-data/spot/): one row per delivery day and slot code 1–48, prices in yen/kWh. The day-ahead auction for day D closes at 10:00 on D−1 ([JEPX guide](https://www.jepx.jp/electricpower/outline/pdf/Guide_2.00.pdf), p. 8), which is why the history rule is 10:30 on D−1.

**Step: parse the current-year file** from today's daily capture, check its range, and size the table if every daily capture is appended whole.

In [8]:
JEPX_COLUMNS = {
    "受渡日": "delivery_date",
    "時刻コード": "slot_code",
    "売り入札量(kWh)": "sell_bid_kwh",
    "買い入札量(kWh)": "buy_bid_kwh",
    "約定総量(kWh)": "contracted_kwh",
    "システムプライス(円/kWh)": "system_price_yen_kwh",
    "エリアプライス北海道(円/kWh)": "price_hokkaido_yen_kwh",
    "エリアプライス東北(円/kWh)": "price_tohoku_yen_kwh",
    "エリアプライス東京(円/kWh)": "price_tokyo_yen_kwh",
    "エリアプライス中部(円/kWh)": "price_chubu_yen_kwh",
    "エリアプライス北陸(円/kWh)": "price_hokuriku_yen_kwh",
    "エリアプライス関西(円/kWh)": "price_kansai_yen_kwh",
    "エリアプライス中国(円/kWh)": "price_chugoku_yen_kwh",
    "エリアプライス四国(円/kWh)": "price_shikoku_yen_kwh",
    "エリアプライス九州(円/kWh)": "price_kyushu_yen_kwh",
    "売りブロック入札総量(kWh)": "sell_block_bid_kwh",
    "売りブロック約定総量(kWh)": "sell_block_contracted_kwh",
    "買いブロック入札総量(kWh)": "buy_block_bid_kwh",
    "買いブロック約定総量(kWh)": "buy_block_contracted_kwh",
}


def parse_jepx(path: str) -> pd.DataFrame:
    df = pd.read_csv(io.StringIO(decode(read(path))), dtype=str)
    unknown = set(df.columns) - set(JEPX_COLUMNS)
    if unknown:
        raise ValueError(f"no ASCII name for columns {sorted(unknown)}")
    df = df.rename(columns=JEPX_COLUMNS).reindex(columns=list(JEPX_COLUMNS.values()))
    df.insert(0, "row_number", range(1, len(df) + 1))
    df["slot_code"] = df["slot_code"].astype(int)
    for column in df.columns.drop(["row_number", "delivery_date", "slot_code"]):
        df[column] = pd.to_numeric(df[column])
    return df.assign(**lineage(path, sidecar(path)))


jepx_daily = files[(files["source"] == "jepx_spot") & (files["load_type"] == "daily")].sort_values(
    "fetched_at"
)
current = parse_jepx(jepx_daily["path"].iloc[-1])
print(
    jepx_daily[["filename", "fetched_jst", "last_modified"]]
    .assign(sha256=jepx_daily["sha256"].str[:10])
    .to_string()
)
print(
    f"\n{len(current)} rows, "
    f"{current['delivery_date'].iloc[0]} to {current['delivery_date'].iloc[-1]}"
)
print(
    "Tokyo price yen/kWh:",
    current["price_tokyo_yen_kwh"].min(),
    "to",
    current["price_tokyo_yen_kwh"].max(),
)
print(
    f"appending each day's file for a year: about {48 * 365 * 365 / 2 / 1e6:.1f} million ODS rows"
)

                filename  fetched_jst last_modified      sha256
0  spot_summary_2026.csv  11-15 11:08           NaN  41ad7a1a2b
1  spot_summary_2026.csv  11-15 11:35           NaN  41ad7a1a2b
2  spot_summary_2026.csv  11-15 12:24           NaN  41ad7a1a2b

11040 rows, 2026/04/01 to 2026/11/16
Tokyo price yen/kWh: 0.01 to 64.28
appending each day's file for a year: about 3.2 million ODS rows


In [9]:
def jepx_rule_time(delivery_date: pd.Series) -> pd.Series:
    """Prices for day D are public after the D−1 auction; 10:30 JST on D−1 is the history rule."""
    day = pd.to_datetime(delivery_date, format="%Y/%m/%d").dt.tz_localize("Asia/Tokyo")
    return day - pd.Timedelta(days=1) + pd.Timedelta(hours=10, minutes=30)


stamped_prices = stamp_known_at(
    current,
    ["delivery_date", "slot_code"],
    jepx_rule_time(current["delivery_date"]),
    CAPTURE_START["jepx_spot"],
)
print("capture start:", CAPTURE_START["jepx_spot"].tz_convert("Asia/Tokyo"))
stamped_prices.tail(2)[
    ["delivery_date", "slot_code", "price_tokyo_yen_kwh", "fetched_at", "known_at", "basis"]
]

capture start: 2026-11-15 11:08:12.892261+09:00


,delivery_date,slot_code,price_tokyo_yen_kwh,fetched_at,known_at,basis
11038,2026/11/16,47,32.11,2026-11-15 03:24:02.979910+00:00,2026-11-15 10:30:00+09:00,rule
11039,2026/11/16,48,31.10,2026-11-15 03:24:02.979910+00:00,2026-11-15 10:30:00+09:00,rule


**What it shows.** **Observed:**

- Three captures today, one content. JEPX sends **no `Last-Modified`** header, so `source_time` is null for JEPX, and a re-capture with `If-Modified-Since` would not work for it (nor is one needed: prices are not revised)
- The 11:08 JST capture already holds **tomorrow** (16 November): the auction result is out by then
- Tokyo prices this fiscal year range from 0.01 yen/kWh (the market's floor) to 64.28
- Appending the whole fiscal-year file every day adds about 3.2 million rows a year. In BigQuery that is a few hundred MB: not worth an optimisation that breaks "one file in, all its rows out"
- Tomorrow's prices are stamped by the rule (10:30 today, before the 11:08 capture). From the 17th on, prices are `captured`

**Learned → design:** `ods.jepx_spot`, wide, as published. The price floor 0.01 is a real value, not an error: the range check is `≥ 0.01`.

## 6. Open-Meteo: one parser, three tables

The three Open-Meteo sources return the same JSON: a list with one entry per point, each with `hourly` arrays. The column names pack three facts into one string: `shortwave_radiation_previous_day2_jma_msm` is variable × lead × model. With a single model, the model suffix is dropped (notebook 01).

**Shape options:**

| Shape | One row per | Rows | Trade-off |
| --- | --- | --- | --- |
| Wide, as returned | Point, hour; a column per variable × lead × model | Fewest | 54 columns whose names hold data; a new model is a schema change |
| Long | Point, model, lead, variable, hour | Most | Uniform, but every row needs a unit, and DWD pivots it back |
| **Semi-long** (proposed) | Point, model, lead (or run), hour; a column per variable | In between | Model and lead become values; variables stay typed columns with fixed units |

The semi-long shape undoes only what JSON forced into the names. The variables really are different columns, each with its own unit.

**Source:** the [forecast API docs](https://open-meteo.com/en/docs): the `timezone` default is "GMT", `wind_speed_unit` defaults to "kmh", radiation is the "average of the preceding hour", and "most weather variables are given as an instantaneous value for the indicated hour". The parser asserts the time zone and every unit, so a silent change of default fails the file instead of corrupting the table.

In [10]:
POINT_NAMES = [name for name, _, _ in WEATHER_POINTS]
OPENMETEO_UNITS = {
    "shortwave_radiation": "W/m²",
    "direct_radiation": "W/m²",
    "diffuse_radiation": "W/m²",
    "cloud_cover": "%",
    "temperature_2m": "°C",
    "wind_speed_10m": "km/h",
}
WEATHER_COLUMNS = list(OPENMETEO_UNITS)


def split_column(column: str, models: list[str]) -> tuple[str, int, str]:
    """'shortwave_radiation_previous_day2_jma_msm' -> ('shortwave_radiation', 2, 'jma_msm')."""
    model = next((m for m in models if column.endswith("_" + m)), None)
    if model is None:
        if len(models) != 1:
            raise ValueError(f"column {column!r} names none of the requested models")
        model, base = models[0], column
    else:
        base = column.removesuffix("_" + model)
    variable, _, lead = base.partition("_previous_day")
    return variable, int(lead or 0), model


def parse_openmeteo(path: str) -> pd.DataFrame:
    """Semi-long: one row per point, model, lead and hour; one column per variable."""
    meta = sidecar(path)
    data = json.loads(read(path))
    data = data if isinstance(data, list) else [data]
    if len(data) != len(POINT_NAMES):
        raise ValueError(f"{len(data)} points, expected {len(POINT_NAMES)}")
    models = meta["params"]["models"].split(",")
    frames = []
    for point, entry in zip(POINT_NAMES, data, strict=True):
        if (entry["timezone"], entry["utc_offset_seconds"]) != ("GMT", 0):
            raise ValueError(f"time zone {entry['timezone']}")
        groups = defaultdict(dict)
        for column, values in entry["hourly"].items():
            if column == "time":
                continue
            variable, lead, model = split_column(column, models)
            if OPENMETEO_UNITS.get(variable) != entry["hourly_units"][column]:
                raise ValueError(f"{column} in {entry['hourly_units'][column]!r}")
            groups[(model, lead)][variable] = values
        for (model, lead), columns in groups.items():
            frame = pd.DataFrame(columns).reindex(columns=WEATHER_COLUMNS).astype(float)
            frame.insert(
                0, "valid_time", pd.to_datetime(entry["hourly"]["time"]).tz_localize("UTC")
            )
            frame.insert(0, "lead_days", lead)
            frame.insert(0, "model", model)
            frame.insert(0, "point", point)
            frame["grid_latitude"], frame["grid_longitude"] = entry["latitude"], entry["longitude"]
            frames.append(frame)
    df = pd.concat(frames, ignore_index=True).dropna(how="all", subset=WEATHER_COLUMNS)
    return df.assign(**lineage(path, meta))


prev_files = files[files["source"] == "openmeteo_previous_runs"].sort_values("filename")
prev_path = prev_files["path"].iloc[-2]
prev = parse_openmeteo(prev_path)
raw_json = json.loads(read(prev_path))
hours_n, columns_n = len(raw_json[0]["hourly"]["time"]), len(raw_json[0]["hourly"]) - 1
print(prev_path)
print(
    f"wide: {len(raw_json)} points x {hours_n} hours x {columns_n} columns "
    f"= {len(raw_json) * hours_n * columns_n:,} cells"
)
print(
    f"long: {int(prev[WEATHER_COLUMNS].notna().sum().sum()):,} rows | semi-long: {len(prev):,} rows"
)
print("\nnon-null hours per point, by model and lead:")
prev.groupby(["model", "lead_days"])[WEATHER_COLUMNS].count().floordiv(len(raw_json))

openmeteo_previous_runs/2026-11-15/20261115T050052Z_previous_runs_2026-10-01_2026-10-31.json
wide: 9 points x 744 hours x 54 columns = 361,584 cells
long: 301,320 rows | semi-long: 60,264 rows

non-null hours per point, by model and lead:


shortwave_radiation  direct_radiation  diffuse_radiation  cloud_cover  temperature_2m  wind_speed_10m
model        lead_days                                                                                                       
ecmwf_ifs025 0                          744               744                744          744             744             744
             1                          744               744                744          744             744             744
             2                          744               744                744          744             744             744
jma_gsm      0                            0                 0                  0          744             744             744
             1                            0                 0                  0          744             744             744
             2                            0                 0                  0          744             744             744
jma_msm      0                          744               744                744          744             744             744
             1                          744               744                744          744             744             744
             2                          744               744                744          744             744             744

**What it shows.** **Observed:**

- One month of Previous Runs is 361,584 cells as returned. The semi-long table needs about a fifth of the long shape's rows, and no unit column
- The non-null counts show each model's coverage: **GSM has no radiation at any lead** (cloud cover, temperature and wind only), so its radiation columns stay null. MSM and ECMWF have all six variables at leads 0, 1 and 2
- The parser accepted the time zone (GMT) and every unit

**Step: `known_at` for Previous Runs.** The response says nothing about when each value was issued ([Previous Runs docs](https://open-meteo.com/en/docs/previous-runs-api)). The PRD's rule: `previous_dayN` was issued about N × 24 h before the valid hour. Adding the model's availability delay makes the stamp conservative whichever run produced the value. The PRD's check that `previous_day2` is safe at `T` already counts that delay; here it becomes part of the stamp. Lead 0 (the "latest" value) gets `valid_time + delay`: it is never knowable a day ahead, which is exactly what the leak scenario F2 needs to show.

In [11]:
MODEL_DELAY = {
    "jma_msm": pd.Timedelta(hours=3),
    "jma_gsm": pd.Timedelta(hours=6),
    "ecmwf_ifs025": pd.Timedelta(hours=6),
}


def previous_runs_known_at(df: pd.DataFrame) -> pd.Series:
    return (
        df["valid_time"]
        - pd.to_timedelta(df["lead_days"] * 24, unit="h")
        + df["model"].map(MODEL_DELAY)
    )


tokyo_noon = prev[
    (prev["point"] == "tokyo") & (prev["valid_time"] == pd.Timestamp("2026-10-15 03:00", tz="UTC"))
]
tokyo_noon.assign(known_at=previous_runs_known_at(tokyo_noon))[
    ["model", "lead_days", "valid_time", "shortwave_radiation", "cloud_cover", "known_at"]
]

,model,lead_days,valid_time,shortwave_radiation,cloud_cover,known_at
339,jma_msm,0,2026-10-15 03:00:00+00:00,530.0,40.0,2026-10-15 06:00:00+00:00
1083,jma_msm,1,2026-10-15 03:00:00+00:00,653.0,9.0,2026-10-14 06:00:00+00:00
1827,jma_msm,2,2026-10-15 03:00:00+00:00,625.0,11.0,2026-10-13 06:00:00+00:00
2571,jma_gsm,0,2026-10-15 03:00:00+00:00,NaN,31.0,2026-10-15 09:00:00+00:00
3315,jma_gsm,1,2026-10-15 03:00:00+00:00,NaN,11.0,2026-10-14 09:00:00+00:00
4059,jma_gsm,2,2026-10-15 03:00:00+00:00,NaN,4.0,2026-10-13 09:00:00+00:00
4803,ecmwf_ifs025,0,2026-10-15 03:00:00+00:00,425.0,99.0,2026-10-15 09:00:00+00:00
5547,ecmwf_ifs025,1,2026-10-15 03:00:00+00:00,635.0,88.0,2026-10-14 09:00:00+00:00
6291,ecmwf_ifs025,2,2026-10-15 03:00:00+00:00,663.0,93.0,2026-10-13 09:00:00+00:00


**Step: Single Runs.** One whole model run per file, named by its start time in the request (`run`). **Source:** the [Single Runs docs](https://open-meteo.com/en/docs/single-runs-api): runs become available "typically 4–6 hours for global models … and 1–3 hours for regional models" after they start, so `known_at = run_time + delay`. A file stored as a recorded gap (`gap: true` in the sidecar) holds Open-Meteo's error message, not data. It must produce no rows; the manifest still records it.

In [12]:
def parse_single_run(path: str) -> pd.DataFrame:
    meta = sidecar(path)
    if meta.get("gap"):
        return pd.DataFrame()
    df = parse_openmeteo(path)
    if df["lead_days"].ne(0).any():
        raise ValueError("a single run has no leads")
    return df.drop(columns="lead_days").assign(
        run_time=pd.Timestamp(meta["params"]["run"], tz="UTC")
    )


single_files = files[files["source"] == "openmeteo_single_runs"]
run = parse_single_run(
    single_files[single_files["filename"] == "jma_msm_2026-09-12T21Z.json"]["path"].iloc[0]
)
run = run.assign(known_at=run["run_time"] + run["model"].map(MODEL_DELAY))
tokyo_hours = run[run["point"] == "tokyo"]["valid_time"]
print(
    f"MSM 21Z run: {len(run):,} rows; Tokyo has {len(tokyo_hours)} hours, "
    f"{tokyo_hours.min()} to {tokyo_hours.max()}"
)
print("recorded gaps among single runs so far:", single_files["gap"].sum())
run[
    ["point", "model", "valid_time", "shortwave_radiation", "cloud_cover", "run_time", "known_at"]
].head(3)

MSM 21Z run: 360 rows; Tokyo has 40 hours, 2026-09-12 21:00:00+00:00 to 2026-09-14 12:00:00+00:00
recorded gaps among single runs so far: 0


,point,model,valid_time,shortwave_radiation,cloud_cover,run_time,known_at
0,tokyo,jma_msm,2026-09-12 21:00:00+00:00,NaN,87.0,2026-09-12 21:00:00+00:00,2026-09-13 00:00:00+00:00
1,tokyo,jma_msm,2026-09-12 22:00:00+00:00,40.0,75.0,2026-09-12 21:00:00+00:00,2026-09-13 00:00:00+00:00
2,tokyo,jma_msm,2026-09-12 23:00:00+00:00,100.0,80.0,2026-09-12 21:00:00+00:00,2026-09-13 00:00:00+00:00


**Step: the live forecast.** Fetched at 08:45 JST each day, so `known_at` is simply `fetched_at`. This is the input the daily forecast will really use at `T`.

In [13]:
def parse_live(path: str) -> pd.DataFrame:
    df = parse_openmeteo(path)
    if df["lead_days"].ne(0).any():
        raise ValueError("the live forecast has no leads")
    return df.drop(columns="lead_days")


live_path = files[(files["source"] == "openmeteo_live") & (files["filename"] == "forecast.json")][
    "path"
].iloc[-1]
live = parse_live(live_path)
print(live_path, f"-> {len(live):,} rows")
live.groupby("model").agg(
    hours=("valid_time", "nunique"),
    first=("valid_time", "min"),
    last=("valid_time", "max"),
    radiation_values=("shortwave_radiation", "count"),
)

openmeteo_live/2026-11-15/20261115T032402Z_forecast.json -> 1,899 rows


,hours,first,last,radiation_values
model,,,,
ecmwf_ifs025,72,2026-11-15 00:00:00+00:00,2026-11-17 23:00:00+00:00,648
jma_gsm,72,2026-11-15 00:00:00+00:00,2026-11-17 23:00:00+00:00,0
jma_msm,67,2026-11-15 00:00:00+00:00,2026-11-17 18:00:00+00:00,603


**What it shows.** **Observed:**

- A short MSM run (21 UTC) covers 40 hours, to 12:00 UTC (21:00 JST) on the 14th. The file holds 168 hours; the rest are null, and the parser drops rows where every variable is null (9 points × 40 hours = 360 rows kept). This matches the PRD's note that short MSM runs end at 21:00 JST on D
- The run's first hour has cloud cover but no radiation: there is no "preceding hour" at the start of a run
- The live file starts at 00:00 UTC today and runs 72 hours; MSM stops five hours short, and GSM has no radiation, as in the history
- No recorded gaps yet: the first 819 runs all fall inside the archive. Tomorrow's resume reaches the holes

**Learned → design:**

- Three tables, one shared parser: `ods.openmeteo_previous_runs` (with `lead_days`), `ods.openmeteo_single_runs` (with `run_time`), `ods.openmeteo_live`. Each keeps the API's own variable names; the units are documented on the table and asserted on every file
- Hour stamps stay as the API gives them, in UTC. **Radiation at hour `t` is the mean of the hour before `t`**: that is a meaning rule, so it belongs in DWD, where hourly weather is placed on 30-minute slots
- `grid_latitude` and `grid_longitude` record the model cell Open-Meteo actually used: 35.7, 139.6875 for the Tokyo point in the three-model request, 35.75, 139.75 for ECMWF alone. Different models snap to different cells

## 7. TEPCO area files: next-day benchmark and same-day files

Five sources share one layout (listed on the [TEPCO download page](https://www.tepco.co.jp/forecast/html/area-download-j.html)). Row 1 names the fields of row 2 (the file's update date and time, and the target day), row 3 is the header, then one row per slot, with `時間コマ` (slot code) 1–48 and three values in kWh per 30 minutes.

| Source | File | What it is |
| --- | --- | --- |
| `tepco_area_forecast_next_day` | `AREA_ONCE_YOSOKU.csv` | TEPCO's forecast for tomorrow: the benchmark |
| `tepco_area_bg_plan_next_day` | `AREA_ONCE_BGKEI.csv` | The balancing groups' plans for tomorrow |
| `tepco_area_forecast` | `AREA_YOSOKU.csv` | Today's forecast |
| `tepco_area_bg_plan` | `AREA_BGKEI.csv` | Today's plans |
| `tepco_area_actuals_today` | `AREA_JISEKI.csv` | Today's actuals so far |

**Step: one parser for all five**, taking columns by position: the headers are spelled differently between files (shown below).

In [14]:
TEPCO_AREA_SOURCES = [
    "tepco_area_forecast_next_day",
    "tepco_area_bg_plan_next_day",
    "tepco_area_forecast",
    "tepco_area_bg_plan",
    "tepco_area_actuals_today",
]
TEPCO_AREA_COLUMNS = [
    "date",
    "slot_code",
    "slot_from",
    "slot_to",
    "demand_kwh",
    "generation_kwh",
    "wind_solar_kwh",
]


def parse_tepco_area(path: str) -> pd.DataFrame:
    lines = decode(read(path)).splitlines()
    if not (lines[0].startswith("ファイル更新日") and lines[2].startswith("日付")):
        raise ValueError("not the area-file layout")
    if len(lines[2].split(",")) != len(TEPCO_AREA_COLUMNS):
        raise ValueError(f"header has {len(lines[2].split(','))} columns")
    updated_date, updated_time, target_date = lines[1].split(",")[:3]
    file_updated = pd.Timestamp(f"{updated_date} {updated_time}", tz="Asia/Tokyo")
    body = "\n".join(lines[3:])
    if body.strip():
        df = pd.read_csv(io.StringIO(body), header=None, names=TEPCO_AREA_COLUMNS, dtype=str)
    else:
        df = pd.DataFrame(columns=TEPCO_AREA_COLUMNS)
    df.insert(0, "row_number", range(1, len(df) + 1))
    df.insert(1, "target_date", target_date)
    df["slot_code"] = df["slot_code"].astype(int)
    for column in ("demand_kwh", "generation_kwh", "wind_solar_kwh"):
        df[column] = pd.to_numeric(df[column])
    return df.assign(**lineage(path, sidecar(path), source_time=file_updated))


rows = []
for source in TEPCO_AREA_SOURCES:
    row = files[files["source"] == source].iloc[-1]
    lines = decode(read(row["path"])).splitlines()
    written = pd.Timestamp(" ".join(lines[1].split(",")[:2]), tz="Asia/Tokyo")
    uploaded = pd.Timestamp(parsedate_to_datetime(row["last_modified"])).tz_convert("Asia/Tokyo")
    rows.append(
        {
            "source": source,
            "written (in file)": f"{written:%m-%d %H:%M:%S}",
            "Last-Modified": f"{uploaded:%m-%d %H:%M:%S}",
            "target": lines[1].split(",")[2],
            "rows": len(parse_tepco_area(row["path"])),
            "header": lines[2][:70],
        }
    )
pd.DataFrame(rows)

,source,written (in file),Last-Modified,target,rows,header
0,tepco_area_forecast_next_day,11-14 23:13:41,11-14 23:20:04,20261115,0,"日付(yyyymmdd),時間コマ(1時～48時),時間帯_自(HH:MI),時間帯_至(H..."
1,tepco_area_bg_plan_next_day,11-14 23:13:41,11-14 23:20:04,20261115,0,"日付(yyyymmdd),時間コマ(1時～48時),時間帯_自(HH:MI),時間帯_至(H..."
2,tepco_area_forecast,11-15 12:12:09,11-15 12:20:04,20261115,48,"日付(yyyymmdd),時間コマ(1時～48時),時間帯_自(HH:MI),時間帯_至(H..."
3,tepco_area_bg_plan,11-15 12:12:09,11-15 12:20:04,20261115,48,"日付(yyyymmdd),時間コマ(1時～48時),時間帯_自(HH:MI),時間帯_至(H..."
4,tepco_area_actuals_today,11-15 12:05:01,11-15 12:10:04,20261115,48,"日付,時間コマ,時間帯＿自,時間帯＿至,エリア総需要量,エリア総発電量,エリア風力・太陽光発電量"


In [15]:
jiseki = parse_tepco_area(files[files["source"] == "tepco_area_actuals_today"]["path"].iloc[-1])
written = jiseki["source_time"].iloc[0]
slot_end = pd.to_datetime(jiseki["date"], format="%Y%m%d").dt.tz_localize(
    "Asia/Tokyo"
) + pd.to_timedelta(jiseki["slot_code"] * 30, unit="min")
future = slot_end > written
all_zero = jiseki[["demand_kwh", "generation_kwh", "wind_solar_kwh"]].eq(0).all(axis=1)
print(
    f"AREA_JISEKI written {written:%H:%M:%S}: {future.sum()} slots end later, "
    f"all zero: {all_zero[future].all()}; "
    f"all-zero slots before it: {all_zero[~future].sum()}"
)

# Units: kWh in 30 minutes / 500 = MW averaged over the slot.
# Compare with the MW file for the same slots.
mw = actuals[actuals["date"] == "2026/11/15"].reset_index(drop=True)
kwh = jiseki[~future].reset_index(drop=True).head(len(mw))
mw = mw.head(len(kwh))
cross = pd.DataFrame(
    {
        "slot": mw["time"],
        "JISEKI demand kWh/500": kwh["demand_kwh"] / 500,
        "eria_jukyu demand MW": mw["area_demand"],
        "JISEKI wind+solar kWh/500": kwh["wind_solar_kwh"] / 500,
        "eria_jukyu wind+solar MW": mw["wind"] + mw["solar"],
    }
)
print(
    f"over {len(cross)} slots, largest difference: "
    f"demand {(cross.iloc[:, 1] - cross.iloc[:, 2]).abs().max():.0f} MW, "
    f"wind+solar {(cross.iloc[:, 3] - cross.iloc[:, 4]).abs().max():.0f} MW"
)
cross.iloc[[0, 12, 20, len(cross) - 1]]

AREA_JISEKI written 12:05:01: 24 slots end later, all zero: True; all-zero slots before it: 0
over 24 slots, largest difference: demand 55 MW, wind+solar 12 MW


,slot,JISEKI demand kWh/500,eria_jukyu demand MW,JISEKI wind+solar kWh/500,eria_jukyu wind+solar MW
0,0:00,23212.0,23189,34.0,42
12,6:00,22648.0,22638,36.0,44
20,10:00,27626.0,27611,3000.0,3012
23,11:30,28668.0,28665,2292.0,2300


**What it shows.** **Observed:**

- **Both next-day files are header-only at midday.** They were rewritten at 23:13:41 JST yesterday for target day 15 November, with no rows. A filled file exists only in the evening, which is why those two sources are captured at 19:00 and 22:00. Tonight's captures will be the first filled ones; their rows should look like `AREA_YOSOKU`'s, which has the identical header
- `AREA_JISEKI` spells its header differently from the other four: no units, a full-width `＿`, times as `0:00` instead of `0000`. Taking columns by position handles that, and the column-count check guards it
- The time written inside each file is 5–8 minutes earlier than its `Last-Modified` header (written, then uploaded). `source_time` uses the time in the file, which is the publisher's own statement
- **`AREA_JISEKI` fills future slots with zeros, not blanks.** Written at 12:05:01, it has 24 slots that end later, all zero, and no zero slot before that time. A zero there means "not yet", which a parser cannot tell from a real zero; DWD can, from `source_time`
- **Units confirmed:** `AREA_JISEKI` in kWh per slot ÷ 500 matches the MW file for the same 24 slots, to within 55 MW on demand (about 0.2%) and 12 MW on wind + solar. Close, but not identical: they are two separate publications of the same quantity, so DWD keeps them as separate series

**Learned → design:**

- One parser, five ODS tables (one per source, as the registry has it). `target_date` and the in-file update time are columns; values stay in kWh per slot
- DWD drops `AREA_JISEKI` slots that end after the file's own update time, and converts kWh per slot ÷ 500 to MW (confirmed above against the MW file)
- `known_at` for the next-day files is `fetched_at`. They are never archived, so there is no history rule: before capture start the benchmark is simply absent (the PRD's approximate fidelity: "the TEPCO benchmark is absent")

## 8. Every raw file through its parser

A parser proven on one sample proves little: notebook 01 found encodings and header layouts that change from file to file. **Step: run every raw file we hold through its parser**, record the result per file, and run the per-file checks. The result is the file manifest, which becomes a table of its own, `ods.raw_files`: one row per raw file, whether or not it produced rows.

Not parsed, by decision: the legacy prefixes; the yearly `area-YYYY.csv` files (phase-1 README: kept raw until a later phase needs them; they differ in unit, grain and header); `tepco_demand_outlook` (`juyo-d1-j.csv`, a multi-section file not used in phase 1). Section 12 later reverses the last two: every captured file becomes a table.

In [16]:
ODS_PARSERS = {
    "tepco_actuals": parse_tepco_actuals,
    "jepx_spot": parse_jepx,
    "openmeteo_previous_runs": parse_openmeteo,
    "openmeteo_single_runs": parse_single_run,
    "openmeteo_live": parse_live,
    **{source: parse_tepco_area for source in TEPCO_AREA_SOURCES},
}
FILE_KEYS = {
    "tepco_actuals": ["date", "time"],
    "jepx_spot": ["delivery_date", "slot_code"],
    "openmeteo_previous_runs": ["point", "model", "lead_days", "valid_time"],
    "openmeteo_single_runs": ["point", "model", "valid_time"],
    "openmeteo_live": ["point", "model", "valid_time"],
    **{source: ["date", "slot_code"] for source in TEPCO_AREA_SOURCES},
}


def check_file(source: str, df: pd.DataFrame) -> str:
    """The per-file checks; in Dagster they become asset checks."""
    issues = []
    duplicates = df.duplicated(FILE_KEYS[source]).sum()
    if duplicates:
        issues.append(f"{duplicates} duplicate keys")
    if source == "tepco_actuals":
        per_day = pd.to_datetime(df["date"], format="%Y/%m/%d").value_counts().sort_index()
        if per_day.iloc[:-1].ne(48).any() or per_day.iloc[-1] > 48:
            issues.append("a day without 48 slots")
        if df["solar"].lt(0).any():
            issues.append("negative solar")
    if source == "jepx_spot" and df.groupby("delivery_date")["slot_code"].nunique().ne(48).any():
        issues.append("a day without 48 slot codes")
    if source in TEPCO_AREA_SOURCES and (
        len(df) not in (0, 48) or set(df["slot_code"]) - set(range(1, 49))
    ):
        issues.append("not 48 slots")
    return "; ".join(issues) or "ok"


def parse_file(row) -> tuple[dict, pd.DataFrame | None]:
    if not row.registered:
        return {"status": "ignored", "note": "legacy prefix"}, None
    if row.gap:
        return {"status": "gap", "note": "recorded gap"}, None
    if row.source == "tepco_demand_outlook" or row.filename.startswith("area-"):
        return {"status": "kept raw", "note": "not parsed in phase 1"}, None
    try:
        df = ODS_PARSERS[row.source](row.path)
    except Exception as e:
        return {"status": "failed", "note": f"{type(e).__name__}: {e}"}, None
    return {
        "status": "parsed" if len(df) else "empty",
        "rows": len(df),
        "check": check_file(row.source, df),
    }, df


results, weather_hours, samples = [], [], {}
for row in files.itertuples():
    result, df = parse_file(row)
    results.append(result)
    if df is not None and len(df):
        samples.setdefault(row.source, df.head(1))
    if row.source == "openmeteo_previous_runs" and df is not None:
        weather_hours.append(
            df[df["point"] == "tokyo"][["model", "lead_days", "valid_time", *WEATHER_COLUMNS]]
        )
manifest = pd.concat(
    [files.drop(columns=["fetched_jst", "registered"]), pd.DataFrame(results)], axis=1
)
manifest["rows"] = manifest["rows"].fillna(0).astype(int)
manifest.groupby(["source", "status"]).agg(files=("path", "count"), rows=("rows", "sum"))

files     rows
source                       status                  
jepx_spot                    parsed       13   208416
openmeteo_live               parsed        2     3798
openmeteo_previous_runs      parsed       35  2016981
openmeteo_single_runs        parsed      819   889497
tepco_actuals                kept raw      8        0
                             parsed       40    53056
tepco_area_actuals_today     parsed        1       48
tepco_area_bg_plan           parsed        1       48
tepco_area_bg_plan_next_day  empty         1        0
tepco_area_forecast          parsed        1       48
tepco_area_forecast_next_day empty         1        0
tepco_demand_outlook         kept raw      1        0
tepco_forecast_next_day      ignored       4        0
tepco_sameday                ignored      20        0

In [17]:
failed = manifest[manifest["status"] == "failed"]
print("failed files:", len(failed))
for row in failed.itertuples():
    print(" ", row.path, "->", row.note)
flagged = manifest[manifest["check"].notna() & manifest["check"].ne("ok")]
print("files whose checks found something:", len(flagged))
for row in flagged.itertuples():
    print(" ", row.path, "->", row.check)

failed files: 0
files whose checks found something: 0


**Step: look for holes in the weather history.** Inside a file, a missing hour is a dropped row (every variable null) or a null in one column. Across all 35 Previous Runs files, at the Tokyo point, find per model, lead and variable every hour missing after that variable's data begins.

In [18]:
tokyo = pd.concat(weather_hours, ignore_index=True)


def first_hour(group: pd.DataFrame, variable: str) -> str:
    hours = group.loc[group[variable].notna(), "valid_time"]
    return f"{hours.min():%Y-%m-%d %H:%M}" if len(hours) else "never"


starts = {
    key: {v: first_hour(group, v) for v in ("shortwave_radiation", "cloud_cover")}
    for key, group in tokyo.groupby(["model", "lead_days"])
}
print("first hour with a value, per model and lead:")
print(pd.DataFrame(starts).T.rename_axis(["model", "lead_days"]).to_string())
holes = []
for (model, lead), group in tokyo.groupby(["model", "lead_days"]):
    series = group.set_index("valid_time").sort_index()[WEATHER_COLUMNS]
    series = series.reindex(pd.date_range(series.index[0], series.index[-1], freq="h"))
    for variable in WEATHER_COLUMNS:
        first = series[variable].first_valid_index()
        if first is None:
            continue  # never provided, like GSM radiation
        missing = series[variable].loc[first:].isna()
        if missing.any():
            span_id = (missing != missing.shift()).cumsum()[missing]
            spans = (
                missing[missing]
                .groupby(span_id)
                .apply(lambda s: f"{s.index[0]:%Y-%m-%d %H:%M} +{len(s)} h")
            )
            holes.append(
                {
                    "model": model,
                    "lead_days": lead,
                    "variable": variable,
                    "missing hours": int(missing.sum()),
                    "holes": "; ".join(spans),
                }
            )
print("\nhours missing after the data begins:")
summary = (
    pd.DataFrame(holes)
    .groupby(["model", "lead_days", "missing hours", "holes"])["variable"]
    .apply(", ".join)
)
print(summary.reset_index().to_string(index=False))

first hour with a value, per model and lead:
                       shortwave_radiation       cloud_cover
model        lead_days                                      
ecmwf_ifs025 0            2024-03-05 08:00  2024-02-03 00:00
             1            2024-03-06 17:00  2024-02-04 00:00
             2            2024-03-07 17:00  2024-02-05 00:00
jma_gsm      0                       never  2024-01-01 00:00
             1                       never  2024-01-01 00:00
             2                       never  2024-01-01 00:00
jma_msm      0            2024-01-01 00:00  2024-01-01 00:00
             1            2024-01-01 00:00  2024-01-01 00:00
             2            2024-01-01 00:00  2024-01-01 00:00

hours missing after the data begins:
       model  lead_days  missing hours                                                                                         holes                                                 variable
ecmwf_ifs025          2            129                  

**What it shows.** **Observed:**

- **Every registered file parsed: 914 files, no failures, and no check found anything.** No duplicate keys, every complete day has 48 slots, no negative solar, every JEPX day has 48 slot codes. That covers all 33 monthly TEPCO history files in both encodings and ten years of JEPX files. It also covers the old `forecast_jma.json` under `openmeteo_live`, which parses because the parser reads the models from the sidecar, not the file name
- The statuses add up to the inventory: 912 parsed, 2 empty (the next-day files at midday), 9 kept raw (eight yearly files and `juyo-d1-j.csv`), 24 ignored
- **The weather history has holes.** The biggest: **MSM has no radiation from 23 January 18:00 UTC to mid-February 2024**, at every lead: 511 hours at lead 0 and 558 at lead 2, which returns on 16 February 00:00 UTC. ECMWF cannot fill it: its cloud cover, temperature and wind begin on 3–5 February 2024 (a day later for each lead), but its radiation only on 5–7 March, which is notebook 01's "spring 2024" made exact. So **no model has radiation from 23 January to 15 February 2024**, and until 7 March lead-2 radiation comes from MSM alone
- MSM at lead 2 also lacks whole hours three times: 5 hours on 12 November 2024, 5 hours on 11 November 2025, 17 hours on 10 November 2026, each from 07:00 UTC
- In April 2026 some variables went missing at leads 1–2: MSM cloud cover and wind for 107 hours from 5 April, GSM cloud cover (lead 1) for 318 hours from 13 April, ECMWF cloud cover (lead 2) for 129 hours from 17 April. MSM radiation was complete throughout

**Learned → design:**

- `ods.raw_files` is the manifest: path, source, extraction date, fetch time, load type, HTTP status, gap, size, sha256, `Last-Modified`, status (`parsed`, `empty`, `gap`, `kept raw`, `ignored`, `failed`), rows, check result, parser version. An empty file or a gap is a fact worth a row: "we looked, and there was nothing"
- A `failed` file fails its partition, and the alert names the file. A wrong ODS table is worse than a late one
- Weather holes are dropped rows in ODS. Finding and filling them (another model, an older lead) is a DWD feature rule, tested in dbt

## 9. `known_at` rules, for DWD

What sections 4 to 7 found, in one table. **Capture start** is a source's first `daily` capture (`CAPTURE_START` above). Every rule is an upper bound: when unsure, later.

| ODS table | First version of a value | Later versions | Basis |
| --- | --- | --- | --- |
| `tepco_actuals` | Slot end + 1 h if that is before capture start, else the first capture showing it | The first capture showing the new value | `rule` before capture start, then `captured` |
| `jepx_spot` | 10:30 JST on D−1 if that is before capture start, else the first capture | Not expected; a change is flagged | `rule`, then `captured` |
| `openmeteo_previous_runs` | Valid hour − 24 h × `lead_days` + model delay (MSM 3 h, GSM and ECMWF 6 h) | – (an archive, fetched once) | `rule` |
| `openmeteo_single_runs` | `run_time` + model delay | – | `rule` |
| `openmeteo_live` | `fetched_at` | – | `captured` |
| `tepco_area_*_next_day` | `fetched_at` | The first capture showing the new value | `captured`; absent before capture start |
| `tepco_actuals_yearly` | Hour end + 1 h (every file ended before capture start) | – | `rule` |
| `tepco_demand_outlook_*` | Today's file: `fetched_at`. The archive: actuals at 18:30 JST on the next day; headline figures at their own update time | The first capture showing the new value | `captured` for today's file, `rule` for the archive |

`as_of(T)` then reads, per key, the latest version with `known_at ≤ T`. A time-machine run's fidelity follows from the rows it used: `exact` only if all of them are `captured`.

## 10. How this maps onto Dagster

| Piece | Design |
| --- | --- |
| Raw assets | One external asset per registered source, partitioned by extraction day (JST). A sensor lists the bucket and records each new file against its source and day |
| ODS assets | One per ODS table, with the same partitions. Materialising day D means "every raw file of D is parsed at the current `parser_version`": files already loaded at that version are skipped, so a re-run adds nothing (idempotent) |
| Re-parsing | Each parser has its own version, used as the asset's code version. A parser change marks every partition stale; a backfill re-parses and **appends** rows under the new version. ODS stays append-only, DWD reads the newest version per file, and a reproduce run can pin the version it used |
| Manifest | The same assets write one `ods.raw_files` row per file and parse |
| Checks | Per file: the parser's refusals and the checks of section 8. Per partition: freshness per source (PRD), and the expected files present (two `tepco_actuals` files a day, one `jepx_spot` file, and so on) |
| BigQuery tables | Partitioned by `extraction_date`, clustered by `source_file`; written by append-only load jobs |
| Code | `jtf/src/jtf/parse/`: one module per layout (`tepco_actuals`, `tepco_area`, `jepx`, `openmeteo`), the lineage helper and the checks. `jtf parse --source S --from D1 --to D2` runs the same code outside Dagster |

A sketch, to be settled in the Dagster spike (step 3):

```python
@asset(partitions_def=extraction_days, code_version=tepco_actuals.PARSER_VERSION, deps=[raw("tepco_actuals")])
def ods_tepco_actuals(context, bigquery, raw_store):
    loaded = bigquery.loaded_files("ods.tepco_actuals", tepco_actuals.PARSER_VERSION)
    for path in raw_store.list(f"tepco_actuals/{context.partition_key}/"):
        if path.endswith(".meta.json") or path in loaded:
            continue
        bigquery.append("ods.tepco_actuals", tepco_actuals.parse(path))
```

## 11. The ODS tables of phase 1

**Step: read the spec off the parsed files**: the columns each table really has, and its size today.

In [19]:
LINEAGE = [
    "source_file",
    "extraction_date",
    "fetched_at",
    "load_type",
    "source_time",
    "parser_version",
]
spec = []
for source, sample in samples.items():
    parsed = manifest[(manifest["source"] == source) & manifest["status"].isin(["parsed", "empty"])]
    columns = [c for c in sample.columns if c not in LINEAGE and c not in FILE_KEYS[source]]
    print(
        f"ods.{source}: {len(parsed)} files, {parsed['rows'].sum():,} rows, "
        f"median {int(parsed['rows'].median()):,} a file"
    )
    print(f"    key within a file: {', '.join(FILE_KEYS[source])}")
    print(f"    other columns: {', '.join(columns)}")
print(f"\nand on every table: {', '.join(LINEAGE)}, plus ingested_at and batch_id at load time")

ods.jepx_spot: 13 files, 208,416 rows, median 17,520 a file
    key within a file: delivery_date, slot_code
    other columns: row_number, sell_bid_kwh, buy_bid_kwh, contracted_kwh, system_price_yen_kwh, price_hokkaido_yen_kwh, price_tohoku_yen_kwh, price_tokyo_yen_kwh, price_chubu_yen_kwh, price_hokuriku_yen_kwh, price_kansai_yen_kwh, price_chugoku_yen_kwh, price_shikoku_yen_kwh, price_kyushu_yen_kwh, sell_block_bid_kwh, sell_block_contracted_kwh, buy_block_bid_kwh, buy_block_contracted_kwh
ods.openmeteo_live: 2 files, 3,798 rows, median 1,899 a file
    key within a file: point, model, valid_time
    other columns: shortwave_radiation, direct_radiation, diffuse_radiation, cloud_cover, temperature_2m, wind_speed_10m, grid_latitude, grid_longitude
ods.openmeteo_previous_runs: 35 files, 2,016,981 rows, median 60,264 a file
    key within a file: point, model, lead_days, valid_time
    other columns: shortwave_radiation, direct_radiation, diffuse_radiation, cloud_cover, temperature_2m, w

**The spec:**

| Table | One row per | Data columns, besides key and lineage | Rows today | Growth |
| --- | --- | --- | --- | --- |
| `ods.tepco_actuals` | Slot (`date`, `time`), per capture | 18 measures in MW averaged over the slot, TEPCO's names in ASCII | 53,056 from 40 files | Two files a day: about 0.8 M rows a year |
| `ods.jepx_spot` | Delivery day and slot code, per capture | Volumes (kWh), the system price and nine area prices (yen/kWh), block volumes | 208,416 from 13 files | About 3.2 M rows a year |
| `ods.openmeteo_previous_runs` | Point, model, `lead_days`, hour | Six variables, grid cell | 2.0 M from 35 files | History only |
| `ods.openmeteo_single_runs` | Point, model, hour, per run (`run_time`) | Six variables, grid cell | 0.9 M from 819 runs | History only; about 1.8 M when the load is done |
| `ods.openmeteo_live` | Point, model, hour, per capture | Six variables, grid cell | 1,899 a capture | About 0.7 M rows a year |
| `ods.tepco_area_forecast_next_day`, `ods.tepco_area_bg_plan_next_day` | Slot code, per capture | `target_date`, `slot_from`, `slot_to`, `demand_kwh`, `generation_kwh`, `wind_solar_kwh` | 0 (header-only at midday) | 48 rows a filled capture, two captures a day |
| `ods.tepco_area_forecast`, `ods.tepco_area_bg_plan`, `ods.tepco_area_actuals_today` | Slot code, per capture | The same | 48 each | 48 rows a capture, if parsed (decision 3) |
| `ods.raw_files` | Raw file | The manifest of section 8 | 947 | One row per file |

In all, about 1 GB a year, inside BigQuery's free 10 GB of storage: the cost here is queries, not storage. Section 12 adds the last two layouts, the yearly actuals and the demand outlook, for 15 tables in all; only the legacy prefixes stay out.

## 12. Every file as a table: the last two layouts

Sections 8 and 11 left two layouts raw: the yearly actuals and the demand outlook. **Decided later on 15 November: every captured file becomes a table**, so feature exploration can query everything we hold. This section designs the two remaining layouts the same way as the others: look, write the parser, run it over every file, check, write the spec.

### 12.1 Yearly actuals: `area-YYYY.csv`

**Source:** the [TEPCO yearly files page](https://www.tepco.co.jp/forecast/html/area_jukyu_p-j.html), FY2016–FY2023. Notebook 01 found the layout: unit row `単位[万kWh]`, a header spread over two rows, thousands separators in FY2016, one thermal column, hourly rows; and that the FY2023 file stops on 31 January 2024, where the monthly files begin.

**Step: survey all eight files** before writing the parser: encoding, unit row, both header rows, and whether numbers are quoted with thousands separators.

In [20]:
def encoding_of(raw: bytes) -> str:
    try:
        raw.decode("utf-8-sig")
        return "utf-8"
    except UnicodeDecodeError:
        return "cp932"


yearly = files[files["filename"].str.startswith("area-")].sort_values("filename")
survey = []
for path in yearly["path"]:
    lines = decode(read(path)).splitlines()
    survey.append(
        {
            "file": path.rsplit("_", 1)[-1],
            "encoding": encoding_of(read(path)),
            "unit row": lines[0].split(",")[0],
            "row 2": lines[1].rstrip(","),
            "row 3 (first 4)": ",".join(lines[2].split(",")[:4]),
            "columns": len(lines[2].split(",")),
            "quoted numbers": any('"' in line for line in lines[3:]),
        }
    )
pd.DataFrame(survey)

,file,encoding,unit row,row 2,row 3 (first 4),columns,quoted numbers
0,area-2016.csv,cp932,単位[万kWh],"DATE,TIME,東京エリア需要,供給力",",,,原子力",15,True
1,area-2017.csv,cp932,単位[万kWh],"DATE,TIME,東京エリア需要,供給力",",,,原子力",15,True
2,area-2018.csv,cp932,単位[万kWh],"DATE,TIME,東京エリア需要,供給力",",,,原子力",15,True
3,area-2019.csv,cp932,単位[万kWh],"DATE,TIME,東京エリア需要,供給力",",,,原子力",15,False
4,area-2020.csv,cp932,単位[万kWh],"DATE,TIME,東京エリア需要,供給力",",,,原子力",15,False
5,area-2021.csv,cp932,単位[万kWh],"DATE,TIME,東京エリア需要,供給力",",,,原子力",15,False
6,area-2022.csv,cp932,単位[万kWh],"DATE,TIME,東京エリア需要,供給力",",,,原子力",15,False
7,area-2023.csv,cp932,単位[万kWh],"DATE,TIME,東京エリア需要,供給力",",,,原子力",15,False


**Step: the parser.** Row 2 names `DATE`, `TIME` and the demand column, then `供給力` ("supply") as a group label; row 3 names the supply columns under it. Joining the two gives one header. A quoted `"2,555"` is a formatting choice, not a different value, so the parser removes the separator and then applies the same whole-number rule as the monthly parser.

In [21]:
import csv

YEARLY_UNIT_ROW = "単位[万kWh]"
YEARLY_COLUMNS = {
    "DATE": "date",
    "TIME": "time",
    "東京エリア需要": "area_demand",
    "原子力": "nuclear",
    "火力": "thermal",
    "水力": "hydro",
    "地熱": "geothermal",
    "バイオマス": "biomass",
    "太陽光発電実績": "solar",
    "太陽光出力制御量": "solar_curtailed",
    "風力発電実績": "wind",
    "風力出力制御量": "wind_curtailed",
    "揚水": "pumped_storage",
    "連系線": "interconnectors",
    "合計": "total",
}
YEARLY_MEASURES = [c for c in YEARLY_COLUMNS.values() if c not in ("date", "time")]


def parse_tepco_actuals_yearly(path: str) -> pd.DataFrame:
    lines = decode(read(path)).splitlines()
    unit = lines[0].split(",")[0]
    if unit != YEARLY_UNIT_ROW:
        raise ValueError(f"unit row is {unit!r}")
    top, below = next(csv.reader([lines[1]])), next(csv.reader([lines[2]]))
    names = top[:3] + below[3:]  # the supply columns sit under 供給力 on row 3
    unknown = sorted(set(names) - set(YEARLY_COLUMNS))
    if unknown:
        raise ValueError(f"no ASCII name for columns {unknown}")
    rows = [r for r in csv.reader(lines[3:]) if any(v.strip() for v in r)]
    df = pd.DataFrame([r[: len(names)] for r in rows], columns=names).rename(columns=YEARLY_COLUMNS)
    df.insert(0, "row_number", range(1, len(df) + 1))
    for column in YEARLY_MEASURES:
        values = df[column].str.replace(",", "", regex=False).replace("", None)
        odd = values.dropna()[~values.dropna().str.fullmatch(r"-?\d+")]
        if len(odd):
            raise ValueError(f"{column} has a value that is not a whole number: {odd.iloc[0]!r}")
        df[column] = pd.to_numeric(values).astype("Int64")
    return df.assign(**lineage(path, sidecar(path)))


checks = []
for path in yearly["path"]:
    df = parse_tepco_actuals_yearly(path)
    day = pd.to_datetime(df["date"], format="%Y/%m/%d")
    checks.append(
        {
            "file": path.rsplit("_", 1)[-1],
            "rows": len(df),
            "first": df["date"].iloc[0],
            "last": df["date"].iloc[-1],
            "hours per day": sorted(day.value_counts().unique()),
            "duplicate hours": int(df.duplicated(["date", "time"]).sum()),
            "solar max (万kWh)": int(df["solar"].max()),
        }
    )
pd.DataFrame(checks)

,file,rows,first,last,hours per day,duplicate hours,solar max (万kWh)
0,area-2016.csv,8760,2016/4/1,2017/3/31,[24],0,788
1,area-2017.csv,8760,2017/4/1,2018/3/31,[24],0,951
2,area-2018.csv,8760,2018/4/1,2019/3/31,[24],0,978
3,area-2019.csv,8784,2019/4/1,2020/3/31,[24],0,1130
4,area-2020.csv,8760,2020/4/1,2021/3/31,[24],0,1431
5,area-2021.csv,8760,2021/4/1,2022/3/31,[24],0,1431
6,area-2022.csv,8760,2022/4/1,2023/3/31,[24],0,1516
7,area-2023.csv,7344,2023/4/1,2024/1/31,[24],0,1602


In [22]:
fy2016 = parse_tepco_actuals_yearly(yearly["path"].iloc[0])
print("raw line:", decode(read(yearly["path"].iloc[0])).splitlines()[3])
fy2016.iloc[:2][
    ["row_number", "date", "time", "area_demand", "thermal", "solar", "total", "load_type"]
]

raw line: 2016/4/1,0:00,"2,555",0,2258,92,0,2,0,0,2,0,0,201,"2,555"


,row_number,date,time,area_demand,thermal,solar,total,load_type
0,1,2016/4/1,0:00,2555,2258,0,2555,initial
1,2,2016/4/1,1:00,2433,2151,0,2433,initial


**What it shows.** **Observed:** 

- All eight files are CP932, with the same unit row (`単位[万kWh]`) and the same two header rows: 15 columns. FY2016–FY2018 quote numbers with thousands separators (`"2,555"`), FY2019 onwards don't
- Every file parses, with 24 hours a day and no duplicate hours: FY2019 has 8,784 rows (29 February 2020), and FY2023 stops at 31 January 2024. 68,688 rows in all
- Solar's highest hour grows from 788 万kWh in FY2016 to 1,602 in FY2023, about 7.9 GW to 16 GW: the capacity growth a solar model has to account for

**Learned → design:**

- `ods.tepco_actuals_yearly`, a second parser under the `tepco_actuals` source. It `reads` the `area-` files; the monthly parser reads the `eria_jukyu_` ones. Each has its own table, because unit, grain and columns differ
- Values stay as published, in 万kWh per hour. Where a unit sits in a unit row, it goes in the table's description and the parser asserts it; where a source names its unit in the header (JEPX, the demand outlook below), the column name carries it. DWD converts: 1 万kWh in an hour is 10 MWh, so 10 MW on average
- `known_at` for these rows is the history rule only (hour end + 1 h, `basis = rule`): the files ended before capture started, and corrections are baked in

### 12.2 The demand outlook: `juyo-d1-j.csv`

The でんき予報 ("electricity forecast") file: today's demand and supply picture for the TEPCO area. We hold one capture, from 12:24 JST today. **Step: split it at its blank lines** and look at what each block is.

In [23]:
def outlook_blocks(text: str) -> list[list[str]]:
    blocks, current = [], []
    for line in text.splitlines():
        if line.strip():
            current.append(line)
        elif current:
            blocks.append(current)
            current = []
    return blocks + ([current] if current else [])


outlook_path = files[files["source"] == "tepco_demand_outlook"]["path"].iloc[-1]
outlook_text = decode(read(outlook_path))
blocks = outlook_blocks(outlook_text)
pd.DataFrame(
    [
        {
            "block": i,
            "lines": len(b),
            "first line": b[0][:70],
            "second line": b[1][:50] if len(b) > 1 else "",
        }
        for i, b in enumerate(blocks)
    ]
)

,block,lines,first line,second line
0,0,3,2026/11/15 12:15 UPDATE,"ピーク時供給力(万kW),時間帯,供給力情報更新日,供給力情報更新時刻,ピーク時予備率(%)..."
1,1,2,"予想最大電力(万kW),時間帯,予想最大電力情報更新日,予想最大電力情報更新時刻","3048,18:00～19:00,11/15,12:10"
2,2,2,"使用率ピーク時供給力(万kW),時間帯,供給力情報更新日,供給力情報更新時刻,使用率ピーク時...","2692,7:00～8:00,11/15,12:10,9,91"
3,3,2,"使用率ピーク時予想電力(万kW),時間帯,予想電力情報更新日,予想電力情報更新時刻","2473,7:00～8:00,11/15,12:10"
4,4,25,"DATE,TIME,当日実績(万kW),予測値(万kW),使用率(%),供給力(万kW)","2026/11/15,0:00,2284,2268,89,2555"
5,5,2,"最大使用率(%),時間帯","92,10:00～11:00"
6,6,2,"翌日のピーク時供給力(万kW),時間帯,供給力情報更新日,供給力情報更新時刻,ピーク時予備率...",",,,,,"
7,7,2,"翌日の予想最大電力(万kW),時間帯,予想最大電力情報更新日,予想最大電力情報更新時刻",",,,"
8,8,2,"翌日の使用率ピーク時供給力(万kW),時間帯,供給力情報更新日,供給力情報更新時刻,使用率ピ...",",,,,,"
9,9,2,"翌日の使用率ピーク時予想電力(万kW),時間帯,予想電力情報更新日,予想電力情報更新時刻",",,,"


**What it shows.** **Observed:** 11 blocks of three shapes:

- **An update line**, `2026/11/15 12:15 UPDATE`, glued to the first block: the file's own update time
- **Nine headline blocks**, a header and one row each: the peak supply, the predicted maximum demand, the supply and demand at the hour of peak usage, the day's maximum usage rate; then the same figures for **tomorrow** (翌日), all empty at 12:24. All in 万kW (*man*-kW, 10⁴ kW = 10 MW) or %, each with its time band and the date and time it was updated
- **An hourly block**: 24 rows of actual demand, TEPCO's demand forecast, usage rate and supply
- **A 5-minute block**: 288 rows of demand, **solar output** and solar's share of demand

**Source:** TEPCO's [explanation of the file](https://www.tepco.co.jp/forecast/html/download-j.html), translated:

- 当日実績: "the hourly average of today's electricity demand. **Until it is confirmed, 0 is entered**"
- 予測値: "the forecast of the hourly average demand for the hours 9:00 to 19:00. 0 is entered once the actual is confirmed"
- "The previous day's actuals are posted every day at about 6:00, then **reviewed, corrected where necessary, and published at about 18:30**"
- "When past demand actuals are corrected, **no notice of the correction is given and the values before correction are not provided**"

**Step: check the zeros** against the file's update time.

In [24]:
updated = pd.Timestamp(blocks[0][0].removesuffix(" UPDATE"), tz="Asia/Tokyo")
hourly_block = next(b for b in blocks if b[0].startswith("DATE,TIME,当日実績(万kW)"))
five_block = next(b for b in blocks if "５分間隔値" in b[0])
hourly = pd.read_csv(io.StringIO("\n".join(hourly_block)), dtype=str)
five = pd.read_csv(io.StringIO("\n".join(five_block)), dtype=str)
hour_end = pd.to_datetime(hourly["DATE"] + " " + hourly["TIME"]).dt.tz_localize(
    "Asia/Tokyo"
) + pd.Timedelta(hours=1)
after = hour_end > updated
print(
    f"updated {updated:%H:%M}; hours ending later: {after.sum()}, "
    f"their actual: {sorted(hourly.loc[after, '当日実績(万kW)'].unique())}"
)
print(
    f"forecasts present: {hourly['予測値(万kW)'].notna().sum()} of 24 hours, "
    "also where the actual is confirmed"
)
blank = five.iloc[:, 2].isna()
print(
    f"5-minute rows: {len(five)}; "
    f"blank from {five.loc[blank, 'TIME'].iloc[0]} on ({blank.sum()} rows)"
)

updated 12:15; hours ending later: 12, their actual: ['0']
forecasts present: 24 of 24 hours, also where the actual is confirmed
5-minute rows: 288; blank from 12:20 on (140 rows)


**Step: is there history for this layout?** The same TEPCO page lists past usage data in two forms: yearly `juyo-YYYY.csv` files up to March 2022, which per the page hold "only the area's total demand", and, from April 2022, a [page of monthly downloads](https://www.tepco.co.jp/forecast/html/download_year-j.html), `YYYYMM_power_usage.zip`. Fetch four months spread over the range (once; cached under `data/probe/`), open the zips and compare each daily file's blocks with today's capture.

In [25]:
import zipfile

import httpx

PROBE = Path("../data/probe/2026-11-15/power_usage")
PROBE.mkdir(parents=True, exist_ok=True)
USAGE_ZIP = "https://www.tepco.co.jp/forecast/html/images/{month}_power_usage.zip"


def usage_zip(month: str) -> zipfile.ZipFile:
    local = PROBE / f"{month}_power_usage.zip"
    if not local.exists():
        response = httpx.get(
            USAGE_ZIP.format(month=month), timeout=60, headers={"User-Agent": "jtf-probe"}
        )
        response.raise_for_status()
        local.write_bytes(response.content)
    return zipfile.ZipFile(local)


def block_heads(text: str) -> tuple[str, ...]:
    return tuple("UPDATE" if b[0].endswith("UPDATE") else b[0] for b in outlook_blocks(text))


live_heads = block_heads(outlook_text)
archive = []
for month in ("202204", "202404", "202507", "202611"):
    archive_zip = usage_zip(month)
    members = archive_zip.namelist()
    texts = {m: archive_zip.read(m).decode("cp932") for m in members}
    tomorrow = [b[1] for b in (outlook_blocks(t)[7] for t in texts.values()) if len(b) > 1]
    archive.append(
        {
            "zip": f"{month}_power_usage.zip",
            "daily files": len(members),
            "first": members[0],
            "same blocks as today's file": all(
                block_heads(t) == live_heads for t in texts.values()
            ),
            "first file's update line": outlook_blocks(texts[members[0]])[0][0],
            "tomorrow's max demand (first file)": tomorrow[0] if tomorrow else "",
        }
    )
pd.DataFrame(archive)

,zip,daily files,first,same blocks as today's file,first file's update line,tomorrow's max demand (first file)
0,202204_power_usage.zip,30,20220401_power_usage.csv,True,2022/4/1 23:55 UPDATE,"3424,18:00～19:00,4/2,0:50"
1,202404_power_usage.zip,30,20240401_power_usage.csv,True,2024/4/1 23:55 UPDATE,"3253,11:00～12:00,4/2,23:50"
2,202507_power_usage.zip,31,20250701_power_usage.csv,True,2025/7/1 23:55 UPDATE,"4893,14:00～15:00,7/2,23:50"
3,202611_power_usage.zip,14,20261101_power_usage.csv,True,2026/11/1 23:55 UPDATE,"3290,16:00～17:00,11/2,23:40"


**What it shows.** **Observed:** 

- Every daily file in all four months, April 2022 to November 2026, has exactly today's 11 blocks: one layout for the whole archive, CP932, one file per day (14 so far in November)
- Each archived file is the **end-of-day version**: its update line says 23:55, so its hourly and 5-minute blocks are complete
- Its "tomorrow" figures were **updated on the day they describe**. The file for 1 November 2026 holds tomorrow's predicted maximum as updated at 23:40 on 2 November; in 2022 the update came at 00:50 on the day itself. The archive keeps only the last version of each figure

**Step: the parser**, one function for both: today's file, and each daily file inside a monthly zip. It returns three tables, one per block shape. The headline blocks become one row each, named by the figure and the day it is for.

In [26]:
OUTLOOK_HOURLY = {
    "DATE": "date",
    "TIME": "time",
    "当日実績(万kW)": "actual_man_kw",
    "予測値(万kW)": "forecast_man_kw",
    "使用率(%)": "usage_pct",
    "供給力(万kW)": "supply_man_kw",
}
OUTLOOK_5MIN = {
    "DATE": "date",
    "TIME": "time",
    "当日実績(５分間隔値)(万kW)": "actual_man_kw",
    "太陽光発電実績(５分間隔値)(万kW)": "solar_man_kw",
    "太陽光発電量(電力使用量に対する割合)(%)": "solar_share_pct",
}
OUTLOOK_HEADLINES = {
    "ピーク時供給力(万kW)": "peak_supply",
    "予想最大電力(万kW)": "peak_demand_forecast",
    "使用率ピーク時供給力(万kW)": "usage_peak_supply",
    "使用率ピーク時予想電力(万kW)": "usage_peak_demand_forecast",
    "最大使用率(%)": "max_usage_pct",
}
HEADLINE_FIELDS = ["value", "time_band", "info_date", "info_time", "reserve_pct", "usage_pct"]


def parse_outlook(text: str, member: str | None = None) -> dict[str, pd.DataFrame]:
    blocks = outlook_blocks(text)
    first = blocks[0]
    if not first[0].endswith(" UPDATE"):
        raise ValueError(f"no update line: {first[0]!r}")
    file_updated = first[0].removesuffix(" UPDATE")
    blocks[0] = first[1:]
    headlines, tables = [], {}
    for block in blocks:
        header = next(csv.reader([block[0]]))
        if header[:2] == ["DATE", "TIME"]:
            mapping = OUTLOOK_HOURLY if header[2] == "当日実績(万kW)" else OUTLOOK_5MIN
            unknown = sorted(set(header) - set(mapping))
            if unknown:
                raise ValueError(f"no ASCII name for columns {unknown}")
            df = pd.read_csv(io.StringIO("\n".join(block)), dtype=str).rename(columns=mapping)
            df.insert(0, "row_number", range(1, len(df) + 1))
            tables["hourly" if mapping is OUTLOOK_HOURLY else "5min"] = df
            continue
        day = "tomorrow" if header[0].startswith("翌日の") else "today"
        figure = OUTLOOK_HEADLINES.get(header[0].removeprefix("翌日の"))
        if figure is None:
            raise ValueError(f"unknown headline block {header[0]!r}")
        values = next(csv.reader([block[1]])) if len(block) > 1 else []
        row = dict(zip(HEADLINE_FIELDS, values + [""] * len(HEADLINE_FIELDS), strict=False))
        headlines.append({"day": day, "figure": figure, **{k: v or None for k, v in row.items()}})
    tables["peaks"] = pd.DataFrame(headlines)
    for df in tables.values():
        df.insert(0, "member", member)
        df["file_updated"] = file_updated
    return tables


today = parse_outlook(outlook_text)
for name, df in today.items():
    print(f"{name}: {len(df)} rows, columns {list(df.columns)}")
today["peaks"][
    ["day", "figure", "value", "time_band", "info_date", "info_time", "reserve_pct", "usage_pct"]
]

hourly: 24 rows, columns ['member', 'row_number', 'date', 'time', 'actual_man_kw', 'forecast_man_kw', 'usage_pct', 'supply_man_kw', 'file_updated']
5min: 288 rows, columns ['member', 'row_number', 'date', 'time', 'actual_man_kw', 'solar_man_kw', 'solar_share_pct', 'file_updated']
peaks: 9 rows, columns ['member', 'day', 'figure', 'value', 'time_band', 'info_date', 'info_time', 'reserve_pct', 'usage_pct', 'file_updated']


,day,figure,value,time_band,info_date,info_time,reserve_pct,usage_pct
0,today,peak_supply,3411,18:00～19:00,11/15,12:10,12,89
1,today,peak_demand_forecast,3048,18:00～19:00,11/15,12:10,NaN,NaN
2,today,usage_peak_supply,2692,7:00～8:00,11/15,12:10,9,91
3,today,usage_peak_demand_forecast,2473,7:00～8:00,11/15,12:10,NaN,NaN
4,today,max_usage_pct,92,10:00～11:00,NaN,NaN,NaN,NaN
5,tomorrow,peak_supply,NaN,NaN,NaN,NaN,NaN,NaN
6,tomorrow,peak_demand_forecast,NaN,NaN,NaN,NaN,NaN,NaN
7,tomorrow,usage_peak_supply,NaN,NaN,NaN,NaN,NaN,NaN
8,tomorrow,usage_peak_demand_forecast,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
parsed_days = {"rows": 0, "files": 0}
nov = usage_zip("202611")
for member in nov.namelist():
    tables = parse_outlook(nov.read(member).decode("cp932"), member)
    parsed_days["files"] += 1
    parsed_days["rows"] += sum(len(t) for t in tables.values())
print(f"all {parsed_days['files']} November daily files parse: {parsed_days['rows']:,} rows")

# Cross-check against the target: 5-minute solar (万kW x 10 = MW) averaged over each 30-minute slot,
# with the 5-minute stamp read as the interval's start, then as its end.
first_day = parse_outlook(nov.read("20261101_power_usage.csv").decode("cp932"), "20261101")["5min"]
stamp = pd.to_datetime(first_day["date"] + " " + first_day["time"])
solar_mw = pd.to_numeric(first_day["solar_man_kw"]) * 10
target = (
    actuals[actuals["date"] == "2026/11/1"]
    .assign(slot=lambda d: pd.to_datetime(d["date"] + " " + d["time"]))
    .set_index("slot")["solar"]
)
for reading, shift in (("start", "0min"), ("end", "5min")):
    slots = (stamp - pd.Timedelta(shift)).dt.floor("30min")
    mean = solar_mw.groupby(slots).mean().reindex(target.index)
    print(
        f"stamp as interval {reading}: "
        f"mean absolute difference {(mean - target).abs().mean():.1f} MW "
        f"over {target.notna().sum()} slots (peak {target.max()} MW)"
    )

all 14 November daily files parse: 4,494 rows
stamp as interval start: mean absolute difference 48.3 MW over 48 slots (peak 13541 MW)
stamp as interval end: mean absolute difference 51.3 MW over 48 slots (peak 13541 MW)


**What it shows.** **Observed:** 

- Today's file gives 9 headline rows (5 for today; the 4 for tomorrow are empty at 12:24), 24 hourly rows and 288 five-minute rows. All 14 November archive files parse: 4,494 rows, 321 a day
- **Units confirmed:** 5-minute solar × 10, averaged over each slot, matches the target file's 30-minute solar to within 48 MW on average on 1 November, about 0.4% of the 13,541 MW peak. Reading the stamp as the interval's start fits slightly better (48 vs 51 MW): too close to settle the alignment on one day, so DWD's design checks it over many
- The forecast column is filled for all 24 hours, also where the actual is confirmed. That is not what TEPCO's explanation describes (see Discrepancies)

**Learned → design:**

- **Three tables for this source**, one per block shape: `ods.tepco_demand_outlook_peaks` (one row per headline figure: `day`, `figure`, `value`, `time_band`, `info_date`, `info_time`, `reserve_pct`, `usage_pct`), `ods.tepco_demand_outlook_hourly` and `ods.tepco_demand_outlook_5min`. Every row carries `member`, the daily file inside a zip (null for today's file), and `file_updated`, the update line (`source_time`)
- **History exists, and it is the same source:** TEPCO's archive of this very file. Like the monthly actuals files, it fits the source's `initial` mode: 56 monthly zips from April 2022, about 80 KB each. The daily capture would add the current and previous month's zip, so each day's final, corrected version arrives the next day; today's file keeps the morning view at 08:45
- **Point-in-time in the archive:** actuals for day X are known at 18:30 JST on X+1 (TEPCO's corrected publication, `basis = rule`). The headline figures carry their own update date and time, which DWD turns into their `known_at`: the archived "tomorrow" figures were updated on the day they describe, so as-of reads never let them in as day-before inputs. That is the F2 lookahead trap, avoided by the data's own timestamps
- Zeros in `actual_man_kw` after `file_updated` mean "not yet", as TEPCO says; DWD drops them, as for `AREA_JISEKI`

### 12.3 The complete ODS spec

| # | Table | Source | One row per | Rows today |
| --- | --- | --- | --- | --- |
| 1 | `ods.tepco_actuals` | `tepco_actuals`, monthly files | Slot, per capture | 53,056 |
| 2 | `ods.tepco_actuals_yearly` | `tepco_actuals`, yearly files | Hour | 68,688 |
| 3–7 | `ods.tepco_area_*` (five tables) | The five `AREA_*` sources | Slot code, per capture | 48 a capture |
| 8 | `ods.tepco_demand_outlook_peaks` | `tepco_demand_outlook` | Headline figure, per file | 9 a file |
| 9 | `ods.tepco_demand_outlook_hourly` | `tepco_demand_outlook` | Hour, per file | 24 a file |
| 10 | `ods.tepco_demand_outlook_5min` | `tepco_demand_outlook` | 5 minutes, per file | 288 a file |
| 11 | `ods.jepx_spot` | `jepx_spot` | Delivery day and slot code, per capture | 208,416 |
| 12–14 | `ods.openmeteo_previous_runs`, `_single_runs`, `_live` | The three Open-Meteo sources | Point, model, lead or run, hour | 2.9 M |
| 15 | `ods.raw_files` | Every source | Raw file and parse | 947 files |

**The rule:** one ODS table per source and file layout, or block shape for a multi-block file. Nothing captured stays raw; only prefixes that aren't registered sources are ignored.

## 13. Learned → design

| Finding | Design |
| --- | --- |
| The extraction date is when a file was fetched; ten years of history sit in one partition | ODS partitions by extraction date (one landing day, one partition); queries filter on the data's own time columns |
| The bucket holds prefixes from before the per-source split | ODS is built from the source registry; legacy prefixes are never parsed |
| Versions and `known_at` need other captures of the same file | ODS changes the format only; versions, `known_at`, units and slot times are DWD rules in dbt |
| `load_type` can't choose the `known_at` rule: a daily capture carries history from before capture start | `known_at` per version: the rule before capture start, captures after it; a `basis` column; a run's fidelity follows from `basis` |
| Identical captures add rows but no versions | Keep every capture; no de-duplication in ODS |
| Shapes differ between files and can change | Parsers refuse unknown columns, unit rows, units, time zones and column counts; a missing column becomes nulls |
| TEPCO publishes a slot about 10 minutes after it ends | The history rule "slot end + 1 h" stays: conservative |
| JEPX sends no `Last-Modified`; the 11:00 file already holds tomorrow | `source_time` is null for JEPX; the daily 11:00 capture is enough |
| Open-Meteo packs model and lead into column names | Semi-long tables with `model` and `lead_days` (or `run_time`); variables keep the API's names; units asserted |
| Radiation is the mean of the preceding hour | DWD places radiation on the hour before its stamp when building 30-minute slots |
| Models snap a point to different grid cells | `grid_latitude` and `grid_longitude` on every row |
| A recorded gap holds an error message | A gap file gives no rows and one manifest row |
| `AREA_*` headers differ; `AREA_JISEKI` zero-fills future slots | One positional parser for five tables; DWD drops slots that end after `source_time` |
| kWh per slot ÷ 500 matches the MW file | DWD's conversion, now checked against data |
| The weather history has holes, one of them three weeks long | Holes stay missing rows and nulls in ODS; a DWD fallback rule; the training window (decision 4) |
| All 914 registered files parse | The parsers move to `jtf/src/jtf/parse/`, tested on small fixtures cut from real files |
| Every file's outcome matters, even an empty one | `ods.raw_files`, one row per file |
| Feature exploration wants every file we hold | 15 ODS tables (section 12.3); nothing captured stays raw |
| The yearly actuals: one layout over FY2016–2023, thousands separators in three years | `ods.tepco_actuals_yearly`, its own parser under `tepco_actuals`; separators removed, 万kWh kept |
| The demand outlook has three block shapes | Three tables, with `member` naming the daily file inside a zip |
| TEPCO archives the demand outlook monthly since April 2022, as end-of-day versions | The source's `initial` items, plus the current and previous month in its daily capture (decision 7) |
| Archived "tomorrow" figures were updated on the day they describe | A headline row's own update time becomes its `known_at` in DWD |
| A parser fix must not rewrite history | Append under a new `parser_version`, which is the Dagster code version |

## Discrepancies

**Against the sources' own statements:**

| Statement | Observed | Effect |
| --- | --- | --- |
| The [Previous Runs docs](https://open-meteo.com/en/docs/previous-runs-api) give archive start dates ("JMA GSM and MSM — available from 2018"; most models "from January 2024") and mention no gaps | MSM radiation is missing at every lead for three weeks of 2024, ECMWF's radiation starts only on 5 March 2024, and shorter holes follow in 2024–2026 (section 8) | The training window and a fallback rule (decision 4) |

| TEPCO's [explanation of the demand outlook](https://www.tepco.co.jp/forecast/html/download-j.html): the forecast covers "the hours 9:00 to 19:00", and "0 is entered once the actual is confirmed" | Today's file has a forecast for all 24 hours, kept after the actual is confirmed; the explanation's example shows an older four-column layout | DWD treats `forecast_man_kw` as published, not as "a forecast only until the actual" |

Everything else the sources state that this notebook could test held: units, time zones, slot codes, the unit rows, and the demand outlook's zeros "until it is confirmed".

**Against our own documents:**

| Document | Says | Found | Change |
| --- | --- | --- | --- |
| Phase-1 README, ingestion | Every ODS row carries `valid_time` and `known_at` | `known_at` depends on other captures; slot times are meaning | Both move to DWD (decision 1) |
| Phase-1 README, ingestion | One `ods.tepco_forecast_next_day` table from `AREA_ONCE_*` | Capture is now split per source | One table per source |
| Phase-1 README, data | The same-day files are "captured only, not parsed yet" | The next-day parser reads them unchanged | Decision 3 |
| Phase-1 README, data | Previous Runs with `models=jma_msm,jma_gsm` | The loader requests all three models, ECMWF included | Fixed in the README |
| Phase-1 README, initial load | ECMWF's leads "from spring 2024" | Radiation from 5–7 March 2024 (leads 0–2); cloud cover, temperature and wind from 3–5 February | Made exact in the README |
| PRD, point-in-time rules | Previous Runs: valid hour − 48 h | The stamp adds the model's delay, as the PRD's own safety check already does | Rule updated in both |
| Phase-1 README and PRD, point-in-time rules | Live weather: the run start "stored as `source_time`" | The forecast API's response names no run (section 6); `source_time` is null for Open-Meteo | Fixed in both |

## Decisions

Agreed on 15 November 2026: the first five as recommended, recorded in the phase-1 README and the PRD; then decisions 6 and 7.

1. **The ODS contract**: format only, with `known_at`, versions, units and slot times in DWD. Recommended
2. **Open-Meteo's shape**: semi-long (recommended), long or wide
3. **The same-day TEPCO area files**: parse them now with the shared parser (recommended: no extra code, and every registered source then has its ODS table) or keep them raw, as the README says
4. **The training window and the weather holes**: start backtests and training on target day **17 February 2024** (recommended: the first day with MSM radiation at lead 2 all day, and seven days of lags behind it; ECMWF radiation, the fallback, joins on 7 March), or on 1 February with no radiation at all for the first two weeks. Either way, a later short hole falls back to ECMWF at the same lead where ECMWF has the variable, and the fallback is recorded per row
5. **Re-parsing**: append under a new `parser_version` (recommended: ODS stays append-only and a reproduce run can pin its version) or overwrite the partition (simpler, but an old replay may no longer reproduce exactly)
6. **Every captured file as a table**, agreed later on 15 November: the yearly actuals and the demand outlook too, for 15 tables (section 12.3)
7. **The demand outlook's history**, agreed: the monthly `YYYYMM_power_usage.zip` archive from April 2022 as the source's `initial` items, and the current and previous month's zip in its daily capture. Recommended: the same pattern as the TEPCO actuals, about 4.5 MB in all, and it brings 5-minute solar and TEPCO's demand forecast back to April 2022 for exploration. Checked before building it: TEPCO answers `If-Modified-Since` on the zips with `304`; this month's zip is missing on the 1st (`404`), so the daily capture takes yesterday's month and the one before; and the zips are rewritten for months (April 2022's was last modified in October 2022), so the monthly re-capture matters

## Sources

| Page | Supports |
| --- | --- |
| [TEPCO area supply and demand, monthly files](https://www.tepco.co.jp/forecast/html/area_jukyu-j.html) | The monthly actuals files; MW averaged per slot |
| [TEPCO area data downloads](https://www.tepco.co.jp/forecast/html/area-download-j.html) | The `AREA_*` files |
| [JEPX spot market page](https://www.jepx.jp/electricpower/market-data/spot/) | Yearly price files |
| [JEPX trading guide](https://www.jepx.jp/electricpower/outline/pdf/Guide_2.00.pdf), p. 8 | Gate closure at 10:00 on D−1 |
| [Open-Meteo forecast API](https://open-meteo.com/en/docs) | Default time zone GMT, wind speed in km/h, radiation as the preceding hour's mean |
| [Open-Meteo Previous Runs API](https://open-meteo.com/en/docs/previous-runs-api) | Lead variables; no issue time in the response |
| [Open-Meteo Single Runs API](https://open-meteo.com/en/docs/single-runs-api) | Run availability delays |